In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Install LightGBM only if it is not already available.
try:
    from lightgbm import LGBMRegressor
except ImportError:
    !pip -q install lightgbm
    from lightgbm import LGBMRegressor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


### Load dataset
The dataset is loaded directly from a public GitHub raw URL, so this notebook does not depend on a separate CSV file being present on the computer or in the repository.


In [ ]:
# Load the dataset directly from GitHub - no local CSV file is required.
DATA_URL = "https://raw.githubusercontent.com/selva86/datasets/master/supermarket_sales.csv"
df = pd.read_csv(DATA_URL)
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df = df.drop(['Invoice ID', 'Date', 'Time', 'cogs', 'gross margin percentage', 'gross income'],axis=1)
label_encoder = LabelEncoder()
df['Branch'] = label_encoder.fit_transform(df['Branch'])
df['City'] = label_encoder.fit_transform(df['City'])
df['Customer type'] = label_encoder.fit_transform(df['Customer type'])
df['Gender'] = label_encoder.fit_transform(df['Gender'])
df['Product line'] = label_encoder.fit_transform(df['Product line'])
df['Payment'] = label_encoder.fit_transform(df['Payment'])

In [ ]:
df.head()

In [ ]:
X = df.drop('Total', axis=1)
y = df['Total']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
model = LGBMRegressor()
model = model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_pred[:10]


In [ ]:

results = pd.DataFrame({
    'Actual Total': y_test.values,
    'Predicted Total': y_pred
})
results.head(10)


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Total")
plt.ylabel("Predicted Total")
plt.title("Actual vs Predicted Total")
plt.show()
